In [1]:
import numba
import numpy as np
import matplotlib.pyplot as plt

@numba.jit(nopython=True)
def sol_iterativa_jacobi(m, tol=1e-8, max_iter=1000000):
    # Usa o método de Jacobi para a solução iterativa do problema
    h = 1 / (m+1)

    x = np.linspace(0, 1, m+2)
    y = np.linspace(0, 1, m+2)

    u = np.zeros((m+2, m+2))
    u[:, 0] = (x-1)*np.sin(x)
    u[:, m+1] = x*(2-x)
    u[0, :] = 0
    u[m+1, :] = y

    iter = 0
    while iter < max_iter:
        u_novo = u.copy()

        for i in range(1, m+1):
            for j in range(1, m+1):
                u_novo[i, j] = (u[i-1, j] + u[i+1, j] + u[i, j-1] + u[i, j+1]) / 4

        matriz_erro = u_novo[1:-1, 1:-1] - u[1:-1, 1:-1]
        erro_relativo = np.linalg.norm(matriz_erro)/np.linalg.norm(u_novo[1:-1, 1:-1])

        u = u_novo

        iter += 1
        if erro_relativo < tol:
            break

    return iter


@numba.jit(nopython=True)
def sol_iterativa_gauss_seidel(m, tol=1e-8, max_iter=1000000):
    # Usa o método de Gauss-Seidel para a solução iterativa do problema
    h = 1 / (m+1)

    x = np.linspace(0, 1, m+2)
    y = np.linspace(0, 1, m+2)

    u = np.zeros((m+2, m+2))
    u[:, 0] = (x-1)*np.sin(x)
    u[:, m+1] = x*(2-x)
    u[0, :] = 0
    u[m+1, :] = y

    iter = 0
    while iter < max_iter:
        u_anterior = u.copy()

        for i in range(1, m+1):
            for j in range(1, m+1):
                u[i, j] = (u[i-1, j] + u[i+1, j] + u[i, j-1] + u[i, j+1]) / 4

        matriz_erro = u[1:-1, 1:-1] - u_anterior[1:-1, 1:-1]

        erro_relativo = np.linalg.norm(matriz_erro) / np.linalg.norm(u[1:-1, 1:-1])

        iter += 1
        if erro_relativo < tol:
            break

    return iter


@numba.jit(nopython=True)
def sol_iterativa_SOR(m, omega, tol=1e-8, max_iter=1000000):
    # Usa SOR (Successive Overrelaxation) para a solução iterativa do problema
    h = 1 / (m+1)

    x = np.linspace(0, 1, m+2)
    y = np.linspace(0, 1, m+2)

    u = np.zeros((m+2, m+2))
    u[:, 0] = (x-1)*np.sin(x)
    u[:, m+1] = x*(2-x)
    u[0, :] = 0
    u[m+1, :] = y

    iter = 0
    while iter < max_iter:
        u_anterior = u.copy()

        for i in range(1, m+1):
            for j in range(1, m+1):
                u_chapeu = (u[i-1, j] + u[i+1, j] + u[i, j-1] + u[i, j+1]) / 4

                delta = u_chapeu - u[i, j]

                u[i, j] = u[i, j] + omega * delta

        matriz_erro = u[1:-1, 1:-1] - u_anterior[1:-1, 1:-1]

        erro_relativo = np.linalg.norm(matriz_erro) / np.linalg.norm(u[1:-1, 1:-1])

        iter += 1
        if erro_relativo < tol:
            break

    return iter

In [2]:
pontos_internos = [19, 39, 79, 159, 319]
omegas = [0.8, 1., 1.5, 1.8, 1.9, 1.95, 1.97]

iteracoes_jacobi = []
iteracoes_gauss_seidel = []
iteracoes_SOR_todas = []
for m in pontos_internos:

    ### Calcula as iterações usando cada método e armazena nas listas
    num_iteracoes_jacobi = sol_iterativa_jacobi(m)
    num_iteracoes_gauss_seidel = sol_iterativa_gauss_seidel(m)

    iteracoes_SOR_omega = []
    for omega in omegas:
        num_iteracoes_SOR = sol_iterativa_SOR(m, omega)
        iteracoes_SOR_omega.append(num_iteracoes_SOR)

    iteracoes_jacobi.append(num_iteracoes_jacobi)
    iteracoes_gauss_seidel.append(num_iteracoes_gauss_seidel)
    iteracoes_SOR_todas.append(iteracoes_SOR_omega)

In [3]:
### Monta a tabela das iterações para convergir de cada $h$ (COM JACOBI)
print('='*34)
print(f'{'Método de Jacobi':^34}')
print('='*34)
print('-'*34)
print(f' {'h':^7} | {'dimensão':^11} {'iterações':^11}')
print('-'*34)
for i in range(len(pontos_internos)):
    m = pontos_internos[i]
    num_iteracoes = iteracoes_jacobi[i]
    dimensao = m**2
    print(f' {'1/'+str(m+1):^7} | {dimensao:^11} {num_iteracoes:^11}')

print('-'*34)

         Método de Jacobi         
----------------------------------
    h    |  dimensão    iterações 
----------------------------------
  1/20   |     361        1090    
  1/40   |    1521        3908    
  1/80   |    6241        13817   
  1/160  |    25281       48033   
  1/320  |   101761      163273   
----------------------------------


In [4]:
### Monta a tabela das iterações para convergir de cada $h$ (COM GAUSS-SEIDEL)
print('='*34)
print(f'{'Método de Gauss Seidel':^34}')
print('='*34)
print('-'*34)
print(f' {'h':^7} | {'dimensão':^11} {'iterações':^11}')
print('-'*34)
for i in range(len(pontos_internos)):
    m = pontos_internos[i]
    num_iteracoes = iteracoes_gauss_seidel[i]
    dimensao = m**2
    print(f' {'1/'+str(m+1):^7} | {dimensao:^11} {num_iteracoes:^11}')

print('-'*34)

      Método de Gauss Seidel      
----------------------------------
    h    |  dimensão    iterações 
----------------------------------
  1/20   |     361         581    
  1/40   |    1521        2082    
  1/80   |    6241        7389    
  1/160  |    25281       25877   
  1/320  |   101761       88953   
----------------------------------


In [5]:
### Monta a tabela das iterações para convergir de cada $h$ (COM SOR)
print('='*67)
print(f'{'Método SOR':^67}')
print('='*67)
print('-'*67)
print(f'{'':^11} | {'1/20':^10} {'1/40':^10} {'1/80':^10} {'1/160':^10} {'1/320':^10}')
print('-'*67)
for i in range(len(omegas)):
    w = omegas[i]

    linha_valores = ''
    for j in range(len(pontos_internos)):
        num_iteracoes = iteracoes_SOR_todas[j][i]
        linha_valores += f' {num_iteracoes:^10}'

    print(f' w = {w:^6.2f} |{linha_valores}')

print('-'*67)

                            Método SOR                             
-------------------------------------------------------------------
            |    1/20       1/40       1/80      1/160      1/320   
-------------------------------------------------------------------
 w =  0.80  |    845        3018      10676      37208      127059  
 w =  1.00  |    581        2082       7389      25877      88953   
 w =  1.50  |    204        756        2714       9611      33529   
 w =  1.80  |     89        252        981        3541      12508   
 w =  1.90  |    180        179        450        1769       6362   
 w =  1.95  |    354        361        355        832        3273   
 w =  1.97  |    589        585        602        643        1940   
-------------------------------------------------------------------
